# Bangladeshi Banknote Detection with YOLOv11

This notebook replaces the supplied people-flow notebook and trains a YOLOv11 model on the requested Kaggle dataset.

**Important limitation:** the Kaggle dataset is organized for classification and has no object-detection bounding boxes. To satisfy the assignment's single-image detection interface, this notebook creates one full-image YOLO box for each banknote image. The resulting model is appropriate for single-note images, but it must not be described as a robust multi-note detector.

Recommended Colab runtime: **T4 GPU**.


In [ ]:
# Cell 1: Install dependencies
!pip -q install -U "ultralytics>=8.3,<9" "kagglehub>=0.3,<1" pyyaml pillow


In [ ]:
# Cell 2: Imports and configuration
from __future__ import annotations

import json
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import torch
import yaml
from PIL import Image, ImageDraw
from ultralytics import YOLO

SEED = 42
random.seed(SEED)

DATASET_HANDLE = "rahnumatasnim1604103/bangladeshi-banknote-dataset"
WORK_DIR = Path("/content/banknote_yolov11")
YOLO_DATASET_DIR = WORK_DIR / "dataset"
RUNS_DIR = WORK_DIR / "runs"

EXPECTED_DENOMINATIONS = ["2", "5", "10", "20", "50", "100", "500", "1000"]
CLASS_NAMES = [f"{value}_taka" for value in EXPECTED_DENOMINATIONS]
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# None uses all images. Use 500 only for a quick pipeline test.
MAX_IMAGES_PER_CLASS = None

EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
CONFIDENCE_THRESHOLD = 0.25

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", "GPU" if DEVICE == 0 else "CPU")
print("Class mapping:", CLASS_TO_ID)


In [ ]:
# Cell 3: Download the public Kaggle dataset
# kagglehub may ask you to authenticate. In Colab, follow the displayed login prompt.
download_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
print("Dataset downloaded to:", download_path)
print("Top-level entries:")
for item in sorted(download_path.iterdir())[:30]:
    print(" -", item.name)


In [ ]:
# Cell 4: Discover images and infer denomination labels from folders
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def denomination_from_text(text: str) -> str | None:
    normalized = re.sub(r"[^0-9a-z]+", " ", text.lower())
    # Longest first prevents 100 from matching 1000.
    for denomination in sorted(EXPECTED_DENOMINATIONS, key=len, reverse=True):
        if re.search(rf"(?<!\d){re.escape(denomination)}(?!\d)", normalized):
            return denomination
    return None

def infer_denomination(image_path: Path, dataset_root: Path) -> str | None:
    relative = image_path.relative_to(dataset_root)
    # Parent folder names are safer than filenames containing sequence numbers.
    for parent_name in reversed(relative.parts[:-1]):
        denomination = denomination_from_text(parent_name)
        if denomination is not None:
            return denomination
    return denomination_from_text(relative.stem)

records_by_class: dict[str, list[Path]] = defaultdict(list)
unmatched_examples: list[str] = []

for image_path in download_path.rglob("*"):
    if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    denomination = infer_denomination(image_path, download_path)
    if denomination is None:
        if len(unmatched_examples) < 20:
            unmatched_examples.append(str(image_path.relative_to(download_path)))
        continue
    class_name = f"{denomination}_taka"
    records_by_class[class_name].append(image_path)

counts = {name: len(records_by_class.get(name, [])) for name in CLASS_NAMES}
print(json.dumps(counts, indent=2))

missing_classes = [name for name, count in counts.items() if count == 0]
if missing_classes:
    print("Unmatched examples:", unmatched_examples)
    raise RuntimeError(f"Could not find images for classes: {missing_classes}")

print("Total matched images:", sum(counts.values()))


In [ ]:
# Cell 5: Create train/validation/test YOLO detection folders
if YOLO_DATASET_DIR.exists():
    shutil.rmtree(YOLO_DATASET_DIR)

for split in ("train", "val", "test"):
    (YOLO_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

split_counts = Counter()
invalid_images: list[str] = []

def split_items(items: list[Path]) -> dict[str, list[Path]]:
    items = items.copy()
    random.Random(SEED).shuffle(items)
    if MAX_IMAGES_PER_CLASS is not None:
        items = items[:MAX_IMAGES_PER_CLASS]

    total = len(items)
    train_end = int(total * TRAIN_RATIO)
    val_end = train_end + int(total * VAL_RATIO)
    return {
        "train": items[:train_end],
        "val": items[train_end:val_end],
        "test": items[val_end:],
    }

for class_name in CLASS_NAMES:
    class_id = CLASS_TO_ID[class_name]
    class_splits = split_items(records_by_class[class_name])

    for split, image_paths in class_splits.items():
        for index, source_path in enumerate(image_paths):
            try:
                with Image.open(source_path) as image:
                    image.verify()
            except Exception:
                invalid_images.append(str(source_path))
                continue

            extension = source_path.suffix.lower()
            destination_name = f"{class_name}_{index:06d}{extension}"
            destination_image = YOLO_DATASET_DIR / "images" / split / destination_name
            destination_label = YOLO_DATASET_DIR / "labels" / split / f"{Path(destination_name).stem}.txt"

            shutil.copy2(source_path, destination_image)
            # YOLO format: class_id center_x center_y width height, normalized to [0, 1].
            # The source dataset provides class folders but no boxes, so each single-note
            # image receives a full-image box.
            destination_label.write_text(f"{class_id} 0.5 0.5 1.0 1.0\n", encoding="utf-8")
            split_counts[(split, class_name)] += 1

print("Invalid images skipped:", len(invalid_images))
for split in ("train", "val", "test"):
    print(f"\n{split.upper()}")
    for class_name in CLASS_NAMES:
        print(f"  {class_name}: {split_counts[(split, class_name)]}")


In [ ]:
# Cell 6: Create data.yaml
data_yaml = {
    "path": str(YOLO_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {index: name for index, name in enumerate(CLASS_NAMES)},
}

DATA_YAML_PATH = YOLO_DATASET_DIR / "data.yaml"
DATA_YAML_PATH.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")
print(DATA_YAML_PATH.read_text())


In [ ]:
# Cell 7: Visualize random converted samples
def draw_full_image_box(image_path: Path, class_name: str) -> Image.Image:
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    draw.rectangle((2, 2, image.width - 3, image.height - 3), outline="red", width=max(2, image.width // 150))
    draw.text((8, 8), class_name, fill="red")
    return image

figure = plt.figure(figsize=(16, 8))
for plot_index, class_name in enumerate(CLASS_NAMES, start=1):
    candidates = list((YOLO_DATASET_DIR / "images" / "train").glob(f"{class_name}_*"))
    sample_path = random.choice(candidates)
    image = draw_full_image_box(sample_path, class_name)
    axis = figure.add_subplot(2, 4, plot_index)
    axis.imshow(image)
    axis.axis("off")
    axis.set_title(class_name)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 8: Train YOLOv11n
model = YOLO("yolo11n.pt")
train_result = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=10,
    workers=2,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="banknote_yolov11",
    exist_ok=True,
    pretrained=True,
    seed=SEED,
    plots=True,
)

BEST_WEIGHTS = Path(model.trainer.save_dir) / "weights" / "best.pt"
LAST_WEIGHTS = Path(model.trainer.save_dir) / "weights" / "last.pt"
print("Best weights:", BEST_WEIGHTS)
print("Last weights:", LAST_WEIGHTS)
assert BEST_WEIGHTS.is_file(), "Training finished without producing best.pt"


In [ ]:
# Cell 9: Evaluate on the held-out test split
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
)

print("mAP50:", float(metrics.box.map50))
print("mAP50-95:", float(metrics.box.map))
print("Mean precision:", float(metrics.box.mp))
print("Mean recall:", float(metrics.box.mr))


In [ ]:
# Cell 10: Single-image inference and JSON output
test_images = sorted((YOLO_DATASET_DIR / "images" / "test").glob("*"))
if not test_images:
    raise RuntimeError("The test split is empty.")

sample_image = random.choice(test_images)
result = best_model.predict(
    source=str(sample_image),
    conf=CONFIDENCE_THRESHOLD,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    verbose=False,
)[0]

detections = []
if result.boxes is not None:
    for coordinates, confidence, class_id in zip(
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
        result.boxes.cls.cpu().numpy().astype(int),
    ):
        x_min, y_min, x_max, y_max = map(float, coordinates)
        detections.append({
            "class_id": int(class_id),
            "denomination": result.names[int(class_id)],
            "confidence": round(float(confidence), 6),
            "bounding_box": {
                "x_min": round(x_min, 2),
                "y_min": round(y_min, 2),
                "x_max": round(x_max, 2),
                "y_max": round(y_max, 2),
            },
        })

print("Input image:", sample_image)
print(json.dumps(detections, indent=2))

annotated_bgr = result.plot()
annotated_rgb = annotated_bgr[:, :, ::-1]
plt.figure(figsize=(12, 7))
plt.imshow(annotated_rgb)
plt.axis("off")
plt.title("Single-image YOLOv11 prediction")
plt.show()

SAMPLE_OUTPUT_PATH = WORK_DIR / "sample_prediction.jpg"
Image.fromarray(annotated_rgb).save(SAMPLE_OUTPUT_PATH)
print("Saved sample output:", SAMPLE_OUTPUT_PATH)


In [ ]:
# Cell 11: Copy deliverables and download them
from google.colab import files

EXPORT_DIR = WORK_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

exported_weights = EXPORT_DIR / "best.pt"
exported_yaml = EXPORT_DIR / "data.yaml"
exported_sample = EXPORT_DIR / "sample_prediction.jpg"

shutil.copy2(BEST_WEIGHTS, exported_weights)
shutil.copy2(DATA_YAML_PATH, exported_yaml)
shutil.copy2(SAMPLE_OUTPUT_PATH, exported_sample)

summary = {
    "classes": CLASS_NAMES,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "mean_precision": float(metrics.box.mp),
    "mean_recall": float(metrics.box.mr),
    "best_weights": str(exported_weights),
}
(EXPORT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

files.download(str(exported_weights))
files.download(str(exported_sample))
files.download(str(EXPORT_DIR / "training_summary.json"))


## After training

1. Put the downloaded `best.pt` into the API project as `models/best.pt`.
2. Add at least five held-out test images to `sample_images/`.
3. Run the API locally or with Docker.
4. Capture actual training, inference, API, and Docker screenshots for the submission.
5. Do not report the example JSON in the README as an actual model result.
